# Sunrise A/B Test on Mock Telco Data

## Business Case: What and Why

Sunrise wants to decide whether a personalized roaming-pack prompt should replace the current app experience for eligible mobile-app customers before peak travel. The decision rule is launch only if the treatment improves 14-day pack conversion and CHF margin while NPS, support contacts, and app opt-outs stay inside pre-defined guardrails.

This notebook is the audit trail for the stakeholder deck in `docs/ab-test-mock-data/deck.md`. The data is synthetic and generated to mimic source-like CRM, billing, app, assignment, and outcome tables.

**Notebook improvements:**
- Inline diagnostic and business charts
- Explicit statistical formulas in LaTeX
- Palette/style audit to confirm Sunrise colors are used in figures

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from sunrise_style import apply_matplotlib_style, color, group_color, status_color, SUNRISE_COLORS

apply_matplotlib_style()

SLUG = "ab-test-mock-data"
RAW = ROOT / "notebooks" / SLUG / "raw"
PROCESSED = ROOT / "notebooks" / SLUG / "processed"
CHARTS = ROOT / "notebooks" / SLUG / "charts"

with (PROCESSED / "ab_test_results.json").open() as handle:
    results = json.load(handle)
results

## EDA: Data Sources, Table Structure, and Wrangling

The mock data uses source-like tables rather than a single flat extract. NPS has explicit missingness and is median-imputed for balance checks and CUPED diagnostics. Outcomes are measured after assignment; pre-period revenue, tenure, app sessions, and NPS are measured before randomization, so they are valid for balance and CUPED.

```mermaid
graph LR
    A[CRM customers: one row per customer] --> F[Joined analysis table]
    B[Billing pre-period: revenue and roaming] --> F
    C[App engagement: sessions and NPS] --> F
    D[Experiment assignment: randomized group] --> F
    E[Experiment outcomes: conversion, margin, guardrails] --> F
    F --> G[Processed summaries and diagnostics]
    G --> H[Charts]
    G --> I[Stakeholder deck]
```


In [ ]:
tables = {
    "crm": pd.read_csv(RAW / "crm_customers.csv"),
    "billing": pd.read_csv(RAW / "billing_preperiod.csv"),
    "app": pd.read_csv(RAW / "app_engagement.csv"),
    "assignment": pd.read_csv(RAW / "experiment_assignment.csv"),
    "outcomes": pd.read_csv(RAW / "experiment_outcomes.csv"),
}

schema_summary = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": table.shape[1], "grain": "customer_id"}
    for name, table in tables.items()
])
schema_summary

In [ ]:
analysis = pd.read_csv(PROCESSED / "analysis_table.csv")
missingness = analysis.isna().mean().sort_values(ascending=False).head(8).rename("missing_rate")
metric_summary = pd.read_csv(PROCESSED / "metric_summary.csv")
balance = pd.read_csv(PROCESSED / "randomization_balance.csv")

display(missingness)
display(metric_summary)
display(balance)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
missingness.plot(kind="bar", color=color("cyan"), ax=ax)
ax.set_title("Top missingness columns in analysis table")
ax.set_ylabel("Missing share")
ax.set_ylim(0, max(0.14, missingness.max() * 1.2))
ax.grid(axis="y")
plt.show()

## Selected Methodology: Theory and Assumptions

The experiment estimates the intent-to-treat impact of showing the prompt. The primary conversion test uses a two-proportion z-test. CHF impact uses a difference in mean 14-day margin per customer and CUPED adjustment:

$$Y_i^{CUPED} = Y_i - \theta(X_i - \bar{X})$$

The conversion test uses:

$$z = \frac{\hat{p}_T - \hat{p}_C}{\sqrt{\hat{p}(1-\hat{p})(1/n_T + 1/n_C)}}$$

and for planning duration we use the standard approximation:

$$n \approx \frac{\left(z_{1-\alpha/2} + z_{\text{power}}\right)^2\cdot 2\cdot p_0(1-p_0)}{\text{MDE}^2}$$

where `Y` is post-treatment 14-day margin and `X` is pre-treatment roaming revenue. Because `X` is measured before randomization, CUPED improves precision without changing the causal estimand.

```mermaid
graph LR
    A[Eligible app customers] --> B{Random assignment}
    B --> C[Control: current app]
    B --> D[Treatment: personalized prompt]
    E[Pre-period roaming revenue] --> F[CUPED adjustment]
    C --> G[Conversion and margin outcomes]
    D --> G
    G --> H[Primary conversion test]
    G --> F
    G --> I[Guardrail tests]
    H --> J[Launch decision]
    F --> J
    I --> J
```


In [ ]:
control = metric_summary.loc[metric_summary.experiment_group == "Control"].iloc[0]
treatment = metric_summary.loc[metric_summary.experiment_group == "Treatment"].iloc[0]

conversion_readout = pd.DataFrame([{
    "control_rate": results["control_conversion_rate"],
    "treatment_rate": results["treatment_conversion_rate"],
    "absolute_lift_pp": 100 * results["conversion_lift_abs"],
    "relative_lift_pct": 100 * results["conversion_lift_rel"],
    "p_value": results["conversion_p_value"],
    "ci_low_pp": 100 * results["conversion_ci_low"],
    "ci_high_pp": 100 * results["conversion_ci_high"],
}])
conversion_readout

In [ ]:
plot_df = metric_summary.copy()
plot_df["rate_pct"] = 100 * plot_df["conversion_rate"]
plot_df = plot_df.set_index("experiment_group").loc[["Control", "Treatment"]].reset_index()
errors = 1.96 * np.sqrt(
    (plot_df["conversion_rate"] * (1 - plot_df["conversion_rate"])) / plot_df["users"]
)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(
    plot_df["experiment_group"],
    plot_df["rate_pct"],
    yerr=100 * errors,
    capsize=6,
    color=[group_color("control"), group_color("treatment")],
)
ax.set_title("14-day conversion with 95% confidence intervals")
ax.set_ylabel("Conversion rate (%)")
ax.grid(axis="y")
for idx, row in plot_df.iterrows():
    ax.text(idx, row["rate_pct"] + 0.15, f"{row['rate_pct']:.2f}%", ha="center", fontweight="bold")
plt.show()

**Chart snapshot (saved artifact):**

![Conversion confidence intervals](ab-test-mock-data/charts/conversion_rate_ci.png)


## Calculation: Tests, Models, Charts, and CHF Impact

The calculation translates the experimental effect into a rollout case using 1.35M annual eligible app customers and repeated 14-day commercial windows. This keeps the unit economics transparent: margin lift per customer per 14 days times eligible reach times periods per year.

In [ ]:
margin_readout = pd.DataFrame([{
    "standard_margin_lift_chf": results["standard_margin_diff_chf"],
    "cuped_margin_lift_chf": results["cuped_margin_diff_chf"],
    "cuped_ci_low_chf": results["cuped_margin_ci_low_chf"],
    "cuped_ci_high_chf": results["cuped_margin_ci_high_chf"],
    "cuped_variance_reduction_pct": 100 * results["cuped_variance_reduction"],
    "annualized_margin_chf": results["annualized_rollout_margin_chf"],
    "annualized_low_chf": results["annualized_rollout_low_chf"],
    "annualized_high_chf": results["annualized_rollout_high_chf"],
}])
margin_readout

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.errorbar(
    ["Standard", "CUPED"],
    [results["standard_margin_diff_chf"], results["cuped_margin_diff_chf"]],
    yerr=[
        [
            results["standard_margin_diff_chf"] - results["standard_margin_ci_low_chf"],
            results["cuped_margin_diff_chf"] - results["cuped_margin_ci_low_chf"],
        ],
        [
            results["standard_margin_ci_high_chf"] - results["standard_margin_diff_chf"],
            results["cuped_margin_ci_high_chf"] - results["cuped_margin_diff_chf"],
        ],
    ],
    fmt="o",
    capsize=8,
    color=color("orange"),
    ecolor=color("blue"),
)
ax.axhline(0, color=color("slate"), linewidth=1)
ax.set_title("CUPED narrows uncertainty around the CHF margin lift")
ax.set_ylabel("Margin lift per customer (CHF, 14-day)")
ax.grid(axis="y")
plt.show()

**Chart snapshot (saved artifact):**

![CUPED margin intervals](ab-test-mock-data/charts/cuped_margin_ci.png)


In [ ]:
guardrails = pd.read_csv(PROCESSED / "guardrails.csv")
power = pd.DataFrame([{
    "mde_pp": 0.6,
    "required_n_per_group": results["required_n_per_group_for_0_6pp_mde"],
    "duration_days_at_8500_daily_traffic": results["duration_days_at_8500_daily_traffic"],
    "observed_power": results["observed_power"],
}])

display(guardrails)
display(power)

In [ ]:
guardrail_plot = guardrails.copy()
guardrail_plot["label"] = guardrail_plot["metric"].map(
    {
        "nps_post": "NPS points",
        "support_contact_14d": "Support contacts (pp)",
        "app_optout_14d": "App opt-out (pp)",
    }
)
display_values = guardrail_plot["difference"].to_numpy().copy()
display_values[1:] = 100 * display_values[1:]
threshold_values = guardrail_plot["threshold"].to_numpy().copy()
threshold_values[1:] = 100 * threshold_values[1:]

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(
    guardrail_plot["label"],
    display_values,
    color=[status_color(status) for status in guardrail_plot["status"]],
)
ax.scatter(threshold_values, guardrail_plot["label"], marker="x", s=110, color=color("coral"), label="Stop threshold")
ax.axvline(0, color=color("slate"), linewidth=1)
ax.set_title("Guardrails remain below stop thresholds")
ax.grid(axis="x")
ax.legend(loc="lower right")
plt.show()

**Chart snapshot (saved artifact):**

![Guardrail status](ab-test-mock-data/charts/guardrails.png)


In [ ]:
# Sunrise palette audit: confirm charts are using shared project colors.
palette_check = pd.Series({
    "control": group_color("control"),
    "treatment": group_color("treatment"),
    "accent_orange": color("orange"),
    "accent_blue": color("blue"),
    "status_pass": status_color("Pass"),
})
display(palette_check)

fig, ax = plt.subplots(figsize=(8, 1.8))
for i, (name, hex_color) in enumerate(palette_check.items()):
    ax.barh([0], [1], left=i, height=0.55, color=hex_color)
    ax.text(i + 0.5, 0, f"{name}: {hex_color}", ha="center", va="center", fontsize=9, color="#0B1F2A")
ax.set_xlim(0, len(palette_check))
ax.set_yticks([])
ax.set_xticks([])
ax.set_title("Palette check against sunrise_style tokens")
for spine in ax.spines.values():
    spine.set_visible(False)
plt.show()

## Results: Conclusion and Recommendation

The treatment wins on the primary metric and clears the business case. Conversion increases by **1.10 pp**, CUPED-adjusted margin increases by **CHF 0.13 per customer per 14 days**, and the annualized rollout case is **CHF 4.7M** with a conservative confidence floor of **CHF 2.0M**.

Recommendation: approve a controlled rollout to 25% of eligible app customers for one week, scale to 50% for one more week, then launch to 100% if NPS, support contacts, and app opt-outs remain green. Keep a 5% post-launch holdout for four weeks to validate persistence and monitor travel-season effects.

Limitations: the data is synthetic, the impact assumes the 14-day margin lift persists across annual travel cycles, and interference between household members is not modeled. A real production test should preregister the metric definitions and freeze the analysis before launch.